In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
writer = SummaryWriter("runs/fake_resnet50")

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [4]:
train_ds = ImageFolder("fake/train", transform=transform)
val_ds   = ImageFolder("fake/val", transform=transform)
test_ds  = ImageFolder("fake/test", transform=transform)

print("Class mapping:", train_ds.class_to_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

Class mapping: {'DeepFakeDetection': 0, 'Deepfakes': 1, 'Face2Face': 2, 'FaceShifter': 3, 'FaceSwap': 4, 'NeuralTextures': 5, 'Real': 6}


In [5]:
class FakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # Freeze everything
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze deeper layers
        for param in self.backbone.layer3.parameters():
            param.requires_grad = True
        for param in self.backbone.layer4.parameters():
            param.requires_grad = True

        in_features = self.backbone.fc.in_features

        # IMPORTANT: No Sigmoid here
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 1)  # raw logits
        )

    def forward(self, x):
        return self.backbone(x)

model = FakeDetector().to(device)

In [6]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

In [7]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)

        # Convert labels properly
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(imgs)

        # HARD CHECK (prevents silent CUDA crash)
        assert outputs.shape == labels.shape, f"{outputs.shape} vs {labels.shape}"

        loss = criterion(outputs, labels)

        if torch.isnan(loss):
            raise ValueError("NaN loss detected")

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)

In [8]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)

            outputs = model(imgs)

            probs = torch.sigmoid(outputs).cpu().numpy().ravel()
            preds = (probs >= 0.5).astype(int)

            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)

    return acc, prec, rec, f1, auc

In [9]:
EPOCHS = 5
best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    acc, prec, rec, f1, auc = evaluate(model, val_loader)

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Accuracy/val", acc, epoch)
    writer.add_scalar("Metrics/precision", prec, epoch)
    writer.add_scalar("Metrics/recall", rec, epoch)
    writer.add_scalar("Metrics/f1", f1, epoch)
    writer.add_scalar("Metrics/auc", auc, epoch)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"loss={train_loss:.4f} | acc={acc:.4f} | "
        f"prec={prec:.4f} | rec={rec:.4f} | f1={f1:.4f} | auc={auc:.4f}"
    )

    if acc > best_val_acc:
        best_val_acc = acc
        torch.save(model.state_dict(), "fake_detector_best.pth")
        print(f"Saved best model (val_acc={acc:.4f})")

writer.close()

ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

In [ ]:
acc, prec, rec, f1, auc = evaluate(model, test_loader)

print(
    f"Accuracy: {acc:.4f} | "
    f"Precision: {prec:.4f} | "
    f"Recall: {rec:.4f} | "
    f"F1: {f1:.4f} | "
    f"AUC: {auc:.4f}"
)